# NLA Steering Playground

The full loop, interactively: **extract → verbalize → edit → reconstruct → steer**.

Everything is Qwen2.5-7B-Instruct, layer 20, matching the released checkpoints.

Reminders:
- This needs an 80 GB GPU with all three models loaded (~42 GB). Run `setup_pod.sh` and `smoke_test.py` first.
- **Stop the pod when you are done.** The volume keeps everything.
- First model load takes 1–2 minutes; after that every step is seconds.

In [ ]:
from nla_lab import NLASession, probes

lab = NLASession()   # loads base model + verbalizer + reconstructor

## 1. Baseline: what does the model do unsteered?

In [ ]:
PROMPT = "Tell me about your day."

baseline = lab.generate(PROMPT, temperature=0.7, seed=0)
print(baseline)

## 2. Extract: read the model's internal state

One layer-20 vector per token. The token list helps you pick a position.
Norms around 100–170 are typical; newlines can spike to ~14,000 and often verbalize poorly.

In [ ]:
acts, toks = lab.extract(PROMPT)
lab.show_tokens(acts, toks, tail=15)

## 3. Verbalize: turn one vector into a sentence

`temperature=0` is reproducible. The fidelity score tells you whether the sentence
actually captured the vector (cos ≥ 0.7 is good; below 0.5, don't trust edits built on it).

In [ ]:
POSITION = len(toks) - 1   # last prompt token; try others

original = lab.verbalize(acts[POSITION], temperature=0.0)
mse, cos = lab.fidelity(original, acts[POSITION])
print(f"fidelity: cos={cos:.3f}  mse={mse:.3f}\n")
print(original)

## 4. Edit the sentence, build the steering direction

The paper's steering vector is a **difference of two reconstructions**:
`direction = AR(edited) − AR(original)`, then `h → h + α·‖h‖·direction`.

Edit tip: keep everything identical except the one property you want to flip.

In [ ]:
# EDIT THIS: start from `original` (printed above) and change ONE thing.
edited = original.replace("formal", "extremely casual")   # <- example; adapt to your text

print("unchanged!" if edited == original else edited)
delta = lab.direction(edited, original)

## 5. Steer: same prompt, sweep the strength

α=0 is the baseline. Too high α gives word salad. Team calibration on Qwen (July 2026, Lambda runs): effects switch on near α=1 for single-position patches and text degrades past ~2; `mode="all"` compounds across positions, so its useful zone sits lower, roughly 0.25–2.

In [ ]:
outs = lab.compare(PROMPT, delta, alphas=(0, 0.25, 0.5, 1, 2), temperature=0.7, seed=0)
for alpha, text in outs.items():
    print(f"──── α={alpha} ────")
    print(text[:400], "\n")

In [ ]:
# judge-free scoring: which measurable behaviors moved?
print(probes.score_table(outs, ["emojis", "exclamations", "ellipses", "contractions", "n_words", "avg_sentence_words"]))

## 6. Free-form directions: your own prompts, no extraction needed

You can skip extract/verbalize entirely: write BOTH sentences yourself and take the difference.
This is the cheapest way to test steering ideas.

**Phrasing matters**: the reconstructor was trained on explanation-style text
(descriptions of a text's style/content/expected continuation), not on instructions.
- in-distribution: *“Casual chat reply packed with emojis, playful upbeat tone.”*
- out-of-distribution: *“Please use lots of emojis.”*

Comparing both phrasings is itself a nice experiment.

In [ ]:
positive = "Casual, excited chat reply packed with emojis and exclamation marks, playful upbeat tone."
negative = "Plain, formal, measured chat reply with no emojis and no exclamation marks, neutral tone."

delta_ff = lab.direction(positive, negative)
outs_ff = lab.compare("Tell me about your day.", delta_ff, alphas=(0, 0.5, 1, 2), temperature=0.7, seed=0)
for alpha, text in outs_ff.items():
    print(f"──── α={alpha} ────")
    print(text[:300], "\n")
print(probes.score_table(outs_ff, ["emojis", "exclamations", "n_words"]))

## Things to try next

- **Graded edits**: “slightly playful” vs “extremely playful” — does α-at-fixed-edit differ from edit-at-fixed-α?
- **Compositional edits**: flip two properties in one sentence; do both behaviors move?
- **Interpolation**: `direction` between two of your own descriptions, applied at increasing α.
- **Steering modes**: `mode="decode"` (prompt untouched) vs `mode="all"` vs `mode="position"` (the paper's single-token setup — see `experiments/repro_rhyme.py`).
- **Fidelity as a diagnostic**: when steering fails, check `lab.fidelity(edited, lab.reconstruct(edited))`-style round trips — did the edit survive reconstruction?

Troubleshooting: mostly-Chinese verbalizer output → injection bug (see README). No `<explanation>` tags → raise `max_new_tokens`. Word salad → lower α.